# Extract V6.1 test-set demo cases

Este notebook genera un set pequeño de imágenes del **test set** para demostrar el modelo en Streamlit.

Extrae ejemplos tipo:

- TP: true positive
- TN: true negative
- FP: false positive
- FN: false negative

Además crea un CSV con `indication`, etiqueta real, probabilidad calibrada, umbral operativo y decisión.


## 1. Montar Drive y configurar rutas

Edita `OUTPUTS_ZIP` si tu ZIP V6.1 está en otra carpeta.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, zipfile, json, re
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# === EDITA ESTAS RUTAS SI ES NECESARIO ===
OUTPUTS_ZIP = Path('/content/drive/MyDrive/Demo/outputs_multimodal_iuxray_v6_1.zip')
STREAMLIT_APP_DIR = Path('/content/streamlit_demo_v6_1/streamlit_demo_v6_1')  # carpeta donde está app.py
EXPORT_DIR = Path('/content/drive/MyDrive/Demo/test_demo_set_v6_1')

# Parámetros de selección
MODEL_NAME = 'image_calibrated'
PROB_COL = 'prob_image_cal'
TARGET_RECALL = 0.90   # usa 0.90 para coherencia con V6.1 final; puedes cambiar a 0.85
N_PER_TYPE = 3         # número de ejemplos por TP/TN/FP/FN

assert OUTPUTS_ZIP.exists(), f'No se encontró el ZIP: {OUTPUTS_ZIP}'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print('ZIP V6.1:', OUTPUTS_ZIP)
print('Export dir:', EXPORT_DIR)
print('Streamlit app dir existe:', STREAMLIT_APP_DIR.exists())

## 2. Leer predicciones del test set desde el ZIP V6.1

No reconstruimos los splits. Usamos directamente `predictions_test_main_v6_1.csv`, que ya contiene solo el test set guardado por el entrenamiento.


In [ ]:
WORK_DIR = Path('/content/v6_1_outputs_tmp')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(OUTPUTS_ZIP, 'r') as z:
    z.extractall(WORK_DIR)

pred_path = WORK_DIR / 'main' / 'predictions_test_main_v6_1.csv'
thr_path = WORK_DIR / 'main' / 'thresholds_main_v6_1.csv'

pred = pd.read_csv(pred_path)
thr = pd.read_csv(thr_path)

# Umbral operativo seleccionado en V6.1
row_thr = thr[
    (thr['model'] == MODEL_NAME) &
    (thr['target_recall'].round(2) == round(TARGET_RECALL, 2))
].copy()
assert len(row_thr) > 0, 'No encontré el umbral solicitado.'
THRESHOLD = float(row_thr.iloc[0]['threshold'])

pred = pred.copy()
pred['prediction'] = (pred[PROB_COL] >= THRESHOLD).astype(int)
pred['case_type'] = np.select(
    [
        (pred['label'] == 1) & (pred['prediction'] == 1),
        (pred['label'] == 0) & (pred['prediction'] == 0),
        (pred['label'] == 0) & (pred['prediction'] == 1),
        (pred['label'] == 1) & (pred['prediction'] == 0),
    ],
    ['TP', 'TN', 'FP', 'FN'],
    default='unknown'
)
pred['threshold'] = THRESHOLD
pred['target_recall'] = TARGET_RECALL

print('Test rows:', pred.shape)
print('Umbral:', THRESHOLD)
print(pred['case_type'].value_counts())
pred[['uid','filename','label','prediction','case_type','diagnostic_category',PROB_COL,'threshold','indication']].head()

## 3. Descargar/localizar el dataset IU-Xray

Las predicciones tienen `filename`, pero las rutas originales pueden venir de Kaggle. Esta celda usa KaggleHub para ubicar las imágenes reales por nombre de archivo.

Si Colab pide login de Kaggle, ejecuta `kagglehub.login()` y vuelve a correr la celda.


In [ ]:
!pip -q install kagglehub
import kagglehub

KAGGLE_DATASET = 'raddar/chest-xrays-indiana-university'
print('Cargando dataset con KaggleHub...')
dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print('Dataset path:', dataset_path)

# Índice filename -> ruta real
image_exts = {'.png', '.jpg', '.jpeg'}
image_index = {p.name: p for p in dataset_path.rglob('*') if p.suffix.lower() in image_exts}
print('Imágenes indexadas:', len(image_index))

missing = [f for f in pred['filename'].unique() if f not in image_index]
print('Imágenes faltantes del test:', len(missing))
if missing[:5]:
    print('Ejemplos faltantes:', missing[:5])

## 4. Seleccionar ejemplos representativos

Criterio usado:

- TP: positivos con mayor probabilidad.
- TN: negativos con menor probabilidad.
- FP: falsos positivos con mayor probabilidad.
- FN: falsos negativos con menor probabilidad.

Puedes cambiar este criterio si quieres casos más moderados o más fáciles de explicar.


In [ ]:
selected_parts = []

for case_type in ['TN', 'TP', 'FP', 'FN']:
    sub = pred[pred['case_type'] == case_type].copy()
    if sub.empty:
        print(f'No hay casos {case_type}')
        continue
    if case_type in ['TP', 'FP']:
        sub = sub.sort_values(PROB_COL, ascending=False)
    else:
        sub = sub.sort_values(PROB_COL, ascending=True)
    selected_parts.append(sub.head(N_PER_TYPE))

selected = pd.concat(selected_parts, ignore_index=True)
selected = selected[selected['filename'].isin(image_index)].copy()
selected['source_image_path'] = selected['filename'].map(lambda f: str(image_index[f]))

cols = [
    'case_type','uid','filename','label','prediction',PROB_COL,'threshold',
    'diagnostic_category','label_tokens','indication','findings','impression','source_image_path'
]
selected[cols]

## 5. Copiar imágenes y crear CSV para la presentación/demo

Esto genera una carpeta con imágenes y un CSV. También crea una copia lista para poner dentro de la carpeta `demo_data` de Streamlit.


In [ ]:
images_dir = EXPORT_DIR / 'images'
images_dir.mkdir(parents=True, exist_ok=True)

records = []
for i, row in selected.iterrows():
    case_type = row['case_type']
    uid = int(row['uid'])
    src = Path(row['source_image_path'])
    safe_name = f"{case_type}_{len([r for r in records if r['case_type']==case_type])+1:02d}_uid{uid}_{src.name}"
    dst = images_dir / safe_name
    shutil.copy2(src, dst)
    records.append({
        'case_type': case_type,
        'uid': uid,
        'filename': row['filename'],
        'image_file': str(Path('images') / safe_name),
        'label': int(row['label']),
        'prediction': int(row['prediction']),
        'probability_calibrated': float(row[PROB_COL]),
        'threshold': float(row['threshold']),
        'target_recall': float(row['target_recall']),
        'diagnostic_category': row.get('diagnostic_category', ''),
        'label_tokens': row.get('label_tokens', ''),
        'indication': row.get('indication', ''),
        'findings': row.get('findings', ''),
        'impression': row.get('impression', ''),
    })

demo_df = pd.DataFrame(records)
csv_path = EXPORT_DIR / 'test_demo_cases_v6_1.csv'
demo_df.to_csv(csv_path, index=False)

# Versión corta para presentación, sin findings/impression si quieres evitar demasiado texto clínico
presentation_df = demo_df[[
    'case_type','uid','image_file','label','prediction','probability_calibrated',
    'threshold','diagnostic_category','label_tokens','indication'
]].copy()
presentation_df.to_csv(EXPORT_DIR / 'test_demo_cases_short_v6_1.csv', index=False)

print('Guardado:', csv_path)
print('Imágenes guardadas en:', images_dir)
demo_df

## 6. Visualizar los casos seleccionados en Colab


In [ ]:
for _, row in demo_df.iterrows():
    img_path = EXPORT_DIR / row['image_file']
    img = Image.open(img_path)
    plt.figure(figsize=(5,5))
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.title(
        f"{row['case_type']} | UID {row['uid']} | y={row['label']} pred={row['prediction']} | "
        f"p={row['probability_calibrated']:.3f} thr={row['threshold']:.3f}"
    )
    plt.show()
    print('Indication:', row['indication'])
    print('Label tokens:', row['label_tokens'])
    print('-' * 100)

## 7. Copiar los ejemplos a la carpeta de Streamlit

Esto no modifica la lógica de predicción. Solo deja las imágenes y el CSV dentro de `demo_data/` para que puedas usarlos durante la exposición o subirlos manualmente en la pestaña Live inference.


In [ ]:
if STREAMLIT_APP_DIR.exists():
    app_demo_dir = STREAMLIT_APP_DIR / 'demo_data' / 'test_demo_cases_v6_1'
    if app_demo_dir.exists():
        shutil.rmtree(app_demo_dir)
    shutil.copytree(EXPORT_DIR, app_demo_dir)
    print('Copiado a Streamlit:', app_demo_dir)
    print('Puedes usar estas imágenes en la demo Live inference subiéndolas desde esa carpeta.')
else:
    print('No encontré STREAMLIT_APP_DIR. No se copió a la app.')

## 8. Crear ZIP descargable del set de prueba


In [ ]:
zip_out = shutil.make_archive(str(EXPORT_DIR), 'zip', EXPORT_DIR)
print('ZIP creado:', zip_out)

# También copia a /content para descargar más fácil desde Colab
content_zip = Path('/content/test_demo_set_v6_1.zip')
shutil.copy2(zip_out, content_zip)
print('Copia descargable:', content_zip)

## Cómo usar este set en la exposición

Recomendación:

1. Abre Streamlit.
2. Ve a **Live inference**.
3. Sube primero un `TN`, luego un `TP`, luego un `FP` o `FN`.
4. Lee el CSV para decir la etiqueta real, la probabilidad calibrada y el umbral.

Frase sugerida:

> These examples come from the held-out test set, not from the training set. I selected true positive, true negative, false positive, and false negative cases to demonstrate both model functionality and limitations.
